# 04 · Graph construction and the mandatory validation gates (all variants, all samples)

For **every** sample, builds the heterogeneous graph under each variant —
{paper Eq. 3 features, official-code features} × {per-sample normalisation, per-dataset normalisation with
the c_max of each split's training set} — and runs every validation gate. The unmodified official model
runs forward + backward on the largest-first sample of each design on the GPU. Graphs are not stored
(≈ 30 GB per variant); they are rebuilt deterministically when needed. Training refuses to start unless
this notebook's report says `passed`. ≈ 15–40 min.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, torch
from verihgn_repro.data import preprocess as P
from verihgn_repro.data.splits import load_manifest
from verihgn_repro import reporting
from verihgn_repro.colab import write_json
SCOPE = 'official9'
PHASH = json.load(open(L.reports / f'03_preprocessing_{SCOPE}.json'))['prep_hash']
PROCESSED = L.processed_for(PHASH)
manifests = {n: load_manifest(L.splits / f'{SCOPE}__{n}.json') for n in ('official_seed42', 'pair_aware')}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
rep = reporting.validate_variants(PROCESSED, manifests, device=device, model_checks_per_design=1)
rep['prep_hash'] = PHASH
write_json(rep, L.reports / f'04_validation_{SCOPE}.json')
write_json([{'variant': k, 'passed': v['failed_count'] == 0 and v['skipped_count'] == 0, **v}
            for k, v in rep['variants'].items()], L.reports / '04_validation_summary.json')
for k, v in rep['variants'].items():
    print(f"{k:55s} ok {v['ok']:4d}  failed {v['failed_count']}  skipped {v['skipped_count']}")
print('c_max per split:', rep['c_max'])
print('official model checks ok:', all(c['ok'] for c in rep['official_model_checks']), len(rep['official_model_checks']))
assert rep['passed'], 'validation failed — training must not start'
print('✔ all validation gates passed for all variants')